# Digital Marketing Campaign Analysis & Revenue Intelligence
### End-to-End EDA, Leakage Audit & Machine Learning Pipeline

**Author:** Samaa Shaikh  
**Dataset:** `online_advertising_performance_data.csv` — 15,408 rows across multi-channel ad campaigns

---

**Notebook Structure**
1. Dataset Loading
2. Data Cleaning
3. Exploratory Data Analysis (EDA)
4. Feature Engineering & Leakage Audit
5. Feature / Target Separation
6. Train / Test Split & Preprocessing
7. Model Training & Evaluation
8. Model Comparison
9. Regression Diagnostics
10. Feature Importance (SHAP)
11. Save Best Pipeline

## 1. Import Libraries

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings('ignore')
print('Libraries loaded successfully.')

## 2. Load Dataset

In [ ]:
# Resolve dataset path whether run from notebooks/ or root
for candidate in [
    "online_advertising_performance_data.csv",
    "../online_advertising_performance_data.csv"
]:
    if os.path.exists(candidate):
        data_path = candidate
        break

df = pd.read_csv(data_path)
df.columns = [c.strip().lower() for c in df.columns]

# Drop empty artifact columns (Unnamed: 12, Unnamed: 13)
df = df.drop(columns=[c for c in df.columns if 'unnamed' in c], errors='ignore')

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

## 3. Data Cleaning

In [ ]:
# Fill missing values
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna('Unknown')
    else:
        df[col] = df[col].fillna(0)

df.replace([np.inf, -np.inf], 0, inplace=True)
print('Missing values after cleaning:', df.isnull().sum().sum())
print(df.dtypes)

## 4. Exploratory Data Analysis (EDA)

EDA is performed on **all columns including revenue-derived business metrics**,  
which is appropriate for understanding the data. These metrics are later **excluded from the ML feature matrix** due to target leakage.

In [ ]:
print('Statistical Summary:')
df.describe()

In [ ]:
# Revenue distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df['revenue'], bins=60, color='#3b82f6', edgecolor='white', alpha=0.8)
axes[0].set_title('Revenue Distribution')
axes[0].set_xlabel('Revenue ($)')

axes[1].hist(df['cost'], bins=60, color='#ef4444', edgecolor='white', alpha=0.8)
axes[1].set_title('Ad Spend (Cost) Distribution')
axes[1].set_xlabel('Cost ($)')

axes[2].hist(df['clicks'], bins=60, color='#10b981', edgecolor='white', alpha=0.8)
axes[2].set_title('Clicks Distribution')
axes[2].set_xlabel('Clicks')

plt.tight_layout()
plt.show()

In [ ]:
# Campaign revenue benchmark
camp_agg = df.groupby('campaign_number').agg(
    Revenue=('revenue', 'sum'),
    Cost=('cost', 'sum'),
).reset_index()
camp_agg['Net Profit'] = camp_agg['Revenue'] - camp_agg['Cost']
camp_agg['ROAS'] = camp_agg['Revenue'] / camp_agg['Cost'].replace(0, np.nan)
print('Campaign Summary (Business Analytics):')
print(camp_agg.to_string(index=False))

In [ ]:
# User engagement breakdown
eng_agg = df.groupby('user_engagement').agg(
    Revenue=('revenue', 'sum'),
    Conversions=('post_click_conversions', 'sum'),
    Clicks=('clicks', 'sum')
).reset_index()
eng_agg['CVR (%)'] = (eng_agg['Conversions'] / eng_agg['Clicks'].replace(0, np.nan)) * 100
print('User Engagement Tier Analysis:')
print(eng_agg.to_string(index=False))

## 5. Feature Engineering & Target Leakage Audit

### Leakage Audit Results

| Feature | Type | Decision | Reason |
|---------|------|----------|---------|
| `roi = (revenue - cost) / cost` | Derived from target | **EXCLUDED** | Contains revenue — direct leakage |
| `post_click_sales_amount` | Revenue proxy | **EXCLUDED** | Post-campaign sales value; revenue proxy |
| `roas = revenue / cost` | Derived from target | **EXCLUDED from ML** | Requires revenue; analytics only |
| `ctr = clicks / displays` | Derived from inputs | **INCLUDED** | Computable from forecast values |
| `cpc = cost / clicks` | Derived from inputs | **INCLUDED** | Computable from forecast values |

> **Note:** ROAS, ROI, Net Profit and Profit Margin are computed below for **business analytics / dashboard use only**. They are explicitly excluded from the ML feature matrix.

In [ ]:
# --- ML-SAFE ENGINEERED FEATURES ---
# Derived from campaign inputs / user-provided forecast values
df['ctr'] = np.where(df['displays'] > 0, df['clicks'] / df['displays'], 0.0)
df['cpc'] = np.where(df['clicks']   > 0, df['cost']   / df['clicks'],   0.0)

# --- BUSINESS ANALYTICS METRICS (EDA/Dashboard ONLY — NOT for ML) ---
# These all require revenue to compute, making them target-derived.
df['roi']           = np.where(df['cost']    > 0, (df['revenue'] - df['cost']) / df['cost'], 0.0)
df['roas']          = np.where(df['cost']    > 0, df['revenue'] / df['cost'], 0.0)
df['net_profit']    = df['revenue'] - df['cost']
df['profit_margin'] = np.where(df['revenue'] > 0, (df['net_profit'] / df['revenue']) * 100, 0.0)

df.replace([np.inf, -np.inf], 0, inplace=True)

print('ML-safe features added: ctr, cpc')
print('Business analytics computed (NOT for ML): roi, roas, net_profit, profit_margin')
df[['displays', 'clicks', 'cost', 'ctr', 'cpc', 'revenue', 'roi', 'roas']].head()

In [ ]:
# Correlation heatmap — ML-safe features vs revenue
corr_cols = ['displays', 'cost', 'clicks', 'post_click_conversions', 'ctr', 'cpc', 'revenue']
plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(df[corr_cols].corr(), dtype=bool))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, linewidths=0.5, square=True, vmin=-1, vmax=1)
plt.title('Pearson Correlation Matrix — ML Feature Space vs. Revenue', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Feature / Target Separation

In [ ]:
# ── ML FEATURE SET (leakage-free) ────────────────────────────────────────────
ML_FEATURES = [
    # Campaign configuration — known before launch
    'month', 'day', 'campaign_number', 'user_engagement', 'banner', 'placement',
    # Campaign inputs — planned / forecasted values
    'displays', 'cost', 'clicks', 'post_click_conversions',
    # Engineered from forecast inputs
    'ctr', 'cpc',
]

# EXCLUDED from ML:
#   roi                     → derived from revenue (target leakage)
#   post_click_sales_amount → post-campaign sales proxy of revenue
#   roas, net_profit, etc.  → business analytics only; all derived from revenue

TARGET = 'revenue'

X = df[ML_FEATURES].copy()
y = df[TARGET].copy()

categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols   = X.select_dtypes(exclude=['object']).columns.tolist()

print(f'Target         : {TARGET}')
print(f'Feature count  : {len(ML_FEATURES)}')
print(f'Categorical    : {categorical_cols}')
print(f'Numerical      : {numerical_cols}')
print(f'X shape        : {X.shape}  |  y shape: {y.shape}')

## 7. Train / Test Split & Preprocessing Pipeline

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Training set : {X_train.shape}  |  Test set : {X_test.shape}')

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('num', StandardScaler(), numerical_cols),
    ],
    remainder='drop'
)
print('Preprocessor: OneHotEncoder (categorical) + StandardScaler (numerical)')

## 8. Model Training & Cross-Validation Evaluation

In [ ]:
models = {
    'Random Forest': RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    'XGBoost':       XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                   random_state=42, verbosity=0),
    'LightGBM':      LGBMRegressor(n_estimators=300, random_state=42, verbose=-1),
    'SVR':           SVR(kernel='rbf', C=10, epsilon=0.1),
}

results     = []
best_model  = None
best_score  = -np.inf
best_name   = ''
best_y_pred = None

for name, estimator in models.items():
    print(f'\n--- Training: {name} ---')
    pipeline = Pipeline([
        ('preprocessing', preprocessor),
        ('model',         estimator),
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    cv_r2 = cross_val_score(pipeline, X, y, cv=5, scoring='r2', n_jobs=-1).mean()

    print(f'  MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}  CV-R2={cv_r2:.4f}')
    results.append({'Model': name, 'MAE': round(mae, 4), 'RMSE': round(rmse, 4),
                    'R2': round(r2, 4), 'CV_R2': round(cv_r2, 4)})

    if r2 > best_score:
        best_score  = r2
        best_model  = pipeline
        best_name   = name
        best_y_pred = y_pred

## 9. Model Comparison

In [ ]:
results_df = pd.DataFrame(results).sort_values('R2', ascending=False).reset_index(drop=True)
print('MODEL COMPARISON:')
print(results_df.to_string(index=False))
print(f'\nBest model (hold-out R2): {best_name}  R2={best_score:.4f}')

# Save results
results_df.to_csv('../outputs/model_comparison.csv', index=False)
print('Saved: outputs/model_comparison.csv')

In [ ]:
# Model comparison bar chart
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2563eb', '#7c3aed', '#059669', '#dc2626']
bars = ax.bar(results_df['Model'], results_df['R2'], color=colors, width=0.5, edgecolor='white')
ax.bar(results_df['Model'], results_df['CV_R2'], color=[c + '55' for c in colors],
       width=0.5, alpha=0.6, label='CV R2 overlay')
for bar, row in zip(bars, results_df.itertuples()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'R2={row.R2:.3f}\nCV={row.CV_R2:.3f}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('R2 Score')
ax.set_title('Model Comparison — Hold-out R2 vs. 5-Fold CV R2 (Leakage-free)', fontweight='bold')
ax.legend(['Hold-out R2', 'CV R2'], loc='lower right')
ax.yaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## 10. Regression Diagnostics — Best Model

In [ ]:
residuals = y_test.values - best_y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predicted vs Actual
axes[0].scatter(y_test, best_y_pred, alpha=0.35, s=12, color='#3b82f6', edgecolors='none')
lim = [min(y_test.min(), best_y_pred.min()) * 0.95,
       max(y_test.max(), best_y_pred.max()) * 1.05]
axes[0].plot(lim, lim, 'r--', linewidth=1.5)
axes[0].set_xlabel('Actual Revenue ($)')
axes[0].set_ylabel('Predicted Revenue ($)')
axes[0].set_title(f'Predicted vs. Actual — {best_name} (R2={best_score:.4f})', fontweight='bold')

# Residuals
axes[1].scatter(best_y_pred, residuals, alpha=0.35, s=12, color='#7c3aed', edgecolors='none')
axes[1].axhline(0, color='red', linewidth=1.5, linestyle='--')
axes[1].set_xlabel('Predicted Revenue ($)')
axes[1].set_ylabel('Residual (Actual - Predicted) ($)')
axes[1].set_title(f'Residual Plot — {best_name}', fontweight='bold')

plt.tight_layout()
plt.show()

## 11. Feature Importance via SHAP

> **Terminology note:** This section shows **feature importance** — how much each input feature  
> contributes to the model's predictions (using SHAP values). This is **not causal attribution**.  
> SHAP values measure statistical contribution to model output, not real-world causation.

In [ ]:
# Fit preprocessor on full dataset for SHAP analysis
_prep = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('num', StandardScaler(), numerical_cols),
    ],
    remainder='drop'
)
X_shap      = _prep.fit_transform(X)
feat_names  = _prep.get_feature_names_out()

xgb_shap    = XGBRegressor(n_estimators=200, random_state=42, verbosity=0)
xgb_shap.fit(X_shap, y)

# Sample 2000 rows for speed
rng       = np.random.default_rng(42)
idx       = rng.choice(X_shap.shape[0], size=min(2000, X_shap.shape[0]), replace=False)
explainer = shap.Explainer(xgb_shap, X_shap[idx])
shap_vals = explainer(X_shap[idx])

shap.summary_plot(shap_vals, X_shap[idx], feature_names=feat_names,
                  max_display=15, show=True)

## 12. Save Best Pipeline

In [ ]:
model_path = '../best_advertising_model.pkl'
joblib.dump(best_model, model_path, compress=3)
print(f'Best model saved: {model_path}')
print(f'Model     : {best_name}')
print(f'Hold-out R2 : {best_score:.4f}')
print(f'Features  : {ML_FEATURES}')